In [1]:
from typing import Optional
from pathlib import Path
import json
import datetime
from urllib.parse import urlparse
import pandas as pd


In [ ]:
def get_data(data_url: str) -> pd.DataFrame:
    """
    Function to download data from a public URL and return it as a pandas DataFrame.
    """
    # Read the CSV file from the URL
    df = pd.read_csv(data_url)
    return df

def extract_url_components(url: str) -> tuple[str, str]:
    
    domain = urlparse(url).netloc
    share_id = urlparse(url).path.split('/')[-1]
    
    return domain, share_id

def build_file_url(
    file_id: str, 
    ext: str, 
    url: Optional[str] = None, 
    domain: Optional[str] = None, 
    share_id: Optional[str] = None
) -> str:
    """Builds a URL for accessing a file on a webdav server.
    If a full URL is provided, it extracts the domain and share_id from it.
    If only domain and share_id are provided, it constructs the URL accordingly.
    If share_id is None, it assumes the file is being uploaded to webdav.
    """
    
    assert (url is not None) or (domain is not None), \
        "Either a full URL or both domain and share_id must be provided."
    if url is not None:
        domain, share_id = extract_url_components(url)
        
    if share_id is None: # When uploading to webdav
        return f"https://{domain}/public.php/webdav/{file_id}.{ext}"
    else: # When downloading from webdav
        return f"https://{domain}/public.php/dav/files/{share_id}/{file_id}.{ext}"


url: str = "https://collab.psa.es/s/WR6MxyJsnZWi9xH"
env_file_id: str = "environment"

request_url = build_file_url(            
    url=url,
    file_id=env_file_id,
    ext="csv"
)
df = get_data(request_url)
df


In [6]:
df.iloc[0]["Unnamed: 0"]


'2022-01-01 00:00:00+00:00'

In [ ]:
from pathlib import Path
import pandas as pd
import datetime
from solhycool_optimization import DayResults

def create_mock_day_results(date_str: str, template_path: Path, template_date_str: str = "20220501") -> DayResults:
    """
    Create a new DayResults object with the same content as the template,
    but with timestamps shifted to match the given date_str.
    """
    # 1. Load from existing file
    original = DayResults.initialize(template_path, date_str=template_date_str)

    # 2. Convert new date_str to datetime
    new_date = pd.to_datetime(date_str, format="%Y%m%d")
    old_date = pd.to_datetime(template_date_str, format="%Y%m%d")
    delta_days = (new_date - old_date).days

    # 3. Shift index and time-dependent fields
    new_index = original.index + pd.Timedelta(days=delta_days)
    new_df_results = original.df_results.copy()
    new_df_results.index = new_df_results.index + pd.Timedelta(days=delta_days)

    # Optional: shift fitness history index if needed (not always datetime-indexed)
    fitness_history = original.fitness_history.copy()
    if isinstance(fitness_history.index, pd.DatetimeIndex):
        fitness_history.index = fitness_history.index + pd.Timedelta(days=delta_days)

    return DayResults(
        index=new_index,
        df_paretos=original.df_paretos,  # usually per-timestep, no timestamp inside
        fitness_history=fitness_history,
        selected_pareto_idxs=original.selected_pareto_idxs,
        df_results=new_df_results,
        consumption_arrays=original.consumption_arrays,
        pareto_idxs=original.pareto_idxs,
        date_str=date_str
    )

# Use current date
mock_date_str = datetime.datetime.today().strftime("%Y%m%d")

mock_day_results = create_mock_day_results(
    date_str=mock_date_str,
    template_path=Path("../dags/results_eval_at_20250421T1741_psa_partial.gz"),
    template_date_str="20220501"
)

# Now you can export or use mock_day_results downstream
mock_day_results.export(Path(f"/tmp/mock_day_results_{mock_date_str}.h5"))


2025-07-21 08:13:30.873 | INFO     | solhycool_optimization:initialize:201 - DayResults loaded for 20220501 from ../dags/results_eval_at_20250421T1741_psa_partial.gz
2025-07-21 08:13:31.402 | INFO     | solhycool_optimization:export:267 - Results for 20250721 saved to /tmp/mock_day_results_20250721.h5


In [6]:
mock_day_results.index[0].dt


AttributeError: 'Timestamp' object has no attribute 'dt'

In [1]:
from pathlib import Path
import hjson
from solhycool_optimization import DayResults
from solhycool_visualization.objects import HorizonResultsVisualizer

plot_config = hjson.loads(Path("../../data/plot_config_day_horizon.hjson").read_text())
day_results = DayResults.initialize(Path("../dags/results_eval_at_20250421T1741_psa_partial.gz"), date_str="20220501")

visualizer = HorizonResultsVisualizer(
    results_plot_config=plot_config,
    day_results=day_results,
)

visualizer.generate_all(
    output_path=Path("."),
    formats=["png"]
)


2025-07-21 12:55:10.112 | INFO     | solhycool_optimization:initialize:201 - DayResults loaded for 20220501 from ../dags/results_eval_at_20250421T1741_psa_partial.gz
2025-07-21 12:55:11.042 | INFO     | phd_visualizations:save_figure:41 - Figure saved in pareto_front_0.png
2025-07-21 12:55:11.259 | INFO     | phd_visualizations:save_figure:41 - Figure saved in pareto_front_1.png
2025-07-21 12:55:11.530 | INFO     | phd_visualizations:save_figure:41 - Figure saved in pareto_front_2.png
2025-07-21 12:55:11.582 | INFO     | phd_visualizations:save_figure:41 - Figure saved in fitness_history_path_selection.png
2025-07-21 12:55:12.704 | INFO     | phd_visualizations:save_figure:41 - Figure saved in results.png
2025-07-21 12:55:12.705 | INFO     | solhycool_visualization.objects:generate_all:128 - Visualizations saved to .


In [1]:
# read_environment task
from deployment import get_data, extract_url_components, build_file_url

url = "https://collab.psa.es/s/WR6MxyJsnZWi9xH"
file_id = "environment"

request_url = build_file_url(            
    url=url,
    file_id=file_id,
    ext="csv"
)

df_env = get_data(request_url)
df_env


,G_Gh,G_Dh,G_Gk,G_Bn,Tamb_C,HR_pct,hs,Az,precip_mm,Td,...,0,Q_kW,Tv_C,mv_kgh,water_price_eur_m3,water_price_alternative_eur_m3,water_price_eur_l,water_price_alternative_eur_l,Vavail_m3,Vavail_l
2024-05-23 00:00:00+00:00,0,0,0,0,18.5,78,0.0,-170.0,0,14.6,...,NaN,200,35,297.774165,0.029054,4.3200,0.000029,0.004320,0.141682,141.681574
2024-05-23 01:00:00+00:00,0,0,0,0,17.3,78,0.0,-173.2,0,13.5,...,NaN,200,35,297.774165,0.029054,5.7040,0.000029,0.005704,0.141682,141.681574
2024-05-23 02:00:00+00:00,0,0,0,0,16.6,78,0.0,-157.3,0,12.8,...,NaN,200,35,297.774165,0.029054,5.6896,0.000029,0.005690,0.141682,141.681574
2024-05-23 03:00:00+00:00,0,0,0,0,16.0,78,0.0,-143.2,0,12.2,...,NaN,200,35,297.774165,0.029054,5.7392,0.000029,0.005739,0.141682,141.681574
2024-05-23 04:00:00+00:00,0,0,0,0,15.5,78,0.0,-131.1,0,11.8,...,NaN,200,35,297.774165,0.029054,5.7600,0.000029,0.005760,0.141682,141.681574
2024-05-23 05:00:00+00:00,0,0,0,0,15.4,65,0.0,-120.5,0,8.8,...,NaN,100,35,148.887083,0.029054,6.4464,0.000029,0.006446,0.141682,141.681574
2024-05-23 06:00:00+00:00,65,32,26,346,15.9,63,5.5,-111.5,0,8.8,...,NaN,100,35,148.887083,0.029054,6.6136,0.000029,0.006614,0.141682,141.681574
2024-05-23 07:00:00+00:00,242,47,111,672,16.9,60,16.9,-103.3,0,9.0,...,NaN,100,35,148.887083,0.029054,5.7392,0.000029,0.005739,0.141682,141.681574
2024-05-23 08:00:00+00:00,446,59,339,805,18.2,55,28.7,-95.1,0,9.0,...,NaN,100,35,148.887083,0.029054,3.9368,0.000029,0.003937,0.141682,141.681574
2024-05-23 09:00:00+00:00,637,83,566,850,19.6,49,40.6,-86.1,0,8.8,...,NaN,100,35,148.887083,0.029054,3.4104,0.000029,0.003410,0.141682,141.681574


In [2]:
# evaluate_optimization task
from dataclasses import asdict
import numpy as np
import pandas as pd
from solhycool_optimization import ValuesDecisionVariables
from solhycool_optimization.problems.horizon.evaluation import evaluate_day
from solhycool_optimization.problems.horizon import AlgoParams

n_parallel_steps: int = 5
values_per_decision_variable: int = 6
algo_params: AlgoParams = AlgoParams()

df_env.index = pd.to_datetime(df_env.index)

# To not spend all day trying
df_env = df_env.iloc[5:8]  # Use only the first day for testing

# date_str = df_env.index[0].strftime("%Y%m%d")
dv_values=ValuesDecisionVariables.initialize(
    values_per_dv=values_per_decision_variable
).generate_arrays()

# print(dv_values)

# Compute total number of evaluations
total_num_evals = np.prod([len(value) for value in asdict(dv_values).values()])

day_results = evaluate_day(
    n_parallel_evals=n_parallel_steps,
    df_day=df_env, 
    dv_values=dv_values, 
    total_num_evals=total_num_evals, 
    path_selector_params=algo_params,
)


2025-07-22 10:33:13.942 | INFO     | solhycool_optimization:initialize:87 - Initializing ValuesDecisionVariables with 6 values per decision variable. A total of 1296 combinations will be generated.
Authorization required, but no authorization protocol specified

Authorization required, but no authorization protocol specified

Authorization required, but no authorization protocol specified

Authorization required, but no authorization protocol specified

2025-07-22 10:33:51.899 | DEBUG    | solhycool_optimization.problems.horizon.evaluation:get_pareto_front:101 - 20240523T0700 | Pareto front indices: [298 299]
2025-07-22 10:33:55.188 | DEBUG    | solhycool_optimization.problems.horizon.evaluation:get_pareto_front:101 - 20240523T0600 | Pareto front indices: [292 293 295]
2025-07-22 10:33:57.955 | DEBUG    | solhycool_optimization.problems.horizon.evaluation:get_pareto_front:101 - 20240523T0500 | Pareto front indices: [285 286 287 288]
2025-07-22 10:33:59.951 | INFO     | solhycool_optimi

In [2]:
from pathlib import Path
from solhycool_optimization import DayResults

template_path = Path("/workspaces/SOLhycool/deployment/dags/results_eval_at_20250421T1741_psa_partial.gz")


day_results = DayResults.initialize(template_path)
day_results.index


2025-07-22 09:41:44.238 | INFO     | solhycool_optimization:initialize:205 - DayResults loaded for 20220501 from /workspaces/SOLhycool/deployment/dags/results_eval_at_20250421T1741_psa_partial.gz


DatetimeIndex(['2022-05-01 00:00:00+00:00', '2022-05-01 01:00:00+00:00',
               '2022-05-01 02:00:00+00:00', '2022-05-01 03:00:00+00:00',
               '2022-05-01 04:00:00+00:00', '2022-05-01 05:00:00+00:00',
               '2022-05-01 06:00:00+00:00', '2022-05-01 07:00:00+00:00',
               '2022-05-01 08:00:00+00:00', '2022-05-01 09:00:00+00:00',
               '2022-05-01 10:00:00+00:00', '2022-05-01 11:00:00+00:00',
               '2022-05-01 12:00:00+00:00', '2022-05-01 13:00:00+00:00',
               '2022-05-01 14:00:00+00:00', '2022-05-01 15:00:00+00:00',
               '2022-05-01 16:00:00+00:00', '2022-05-01 17:00:00+00:00',
               '2022-05-01 18:00:00+00:00', '2022-05-01 19:00:00+00:00',
               '2022-05-01 20:00:00+00:00', '2022-05-01 21:00:00+00:00',
               '2022-05-01 22:00:00+00:00', '2022-05-01 23:00:00+00:00'],
              dtype='datetime64[ns, UTC]', name='time', freq=None)

In [6]:
import pandas as pd

ds = pd.Series(day_results.df_paretos, index=day_results.index).head(10)


In [7]:
ds.iloc[0]


,Tamb,HR,mv,Tv,qc,Tc_in,Tc_out,Tcond,Qc_released,Qc_absorbed,...,Vavail_s1,Je,Jw,J,Je_c,Je_dc,Je_wct,Jw_wct,Jw_s1,Jw_s2
0,11.5,77.0,297.774165,35.0,22.050611,25.186036,32.813964,35.0,200.0,195.302011,...,0.770662,1.741826,0.000658,1.742485,0.741126,0.979181,0.021520,0.000658,0.000658,0.0
1,11.5,77.0,297.774165,35.0,22.050611,25.186036,32.813964,35.0,200.0,195.302011,...,0.770662,1.530454,0.000796,1.531250,0.741126,0.767881,0.021448,0.000796,0.000796,0.0
2,11.5,77.0,297.774165,35.0,22.050611,25.186036,32.813964,35.0,200.0,195.302011,...,0.770662,1.530452,0.001055,1.531507,0.741126,0.767881,0.021446,0.001055,0.001055,0.0
3,11.5,77.0,297.774165,35.0,22.050611,25.186036,32.813964,35.0,200.0,195.302011,...,0.770662,1.316125,0.001073,1.317198,0.741126,0.552992,0.022007,0.001073,0.001073,0.0
4,11.5,77.0,297.774165,35.0,22.050611,25.186036,32.813964,35.0,200.0,195.302011,...,0.770662,1.315893,0.001145,1.317038,0.741126,0.552992,0.021775,0.001145,0.001145,0.0
5,11.5,77.0,297.774165,35.0,19.946922,24.680971,33.319029,35.0,200.0,200.065193,...,0.770662,1.152468,0.001212,1.153680,0.576072,0.552992,0.023404,0.001212,0.001212,0.0
6,11.5,77.0,297.774165,35.0,17.843233,24.171764,33.828236,35.0,200.0,200.065193,...,0.770662,1.014439,0.001240,1.015679,0.436383,0.552992,0.025064,0.001240,0.001240,0.0
7,11.5,77.0,297.774165,35.0,17.843233,24.171764,33.828236,35.0,200.0,200.065193,...,0.770662,1.014246,0.001549,1.015795,0.436383,0.552992,0.024871,0.001549,0.001549,0.0
8,11.5,77.0,297.774165,35.0,17.843233,24.171764,33.828236,35.0,200.0,200.065193,...,0.770662,1.013335,0.001567,1.014902,0.436383,0.552992,0.023960,0.001567,0.001567,0.0
9,11.5,77.0,297.774165,35.0,17.843233,24.171764,33.828236,35.0,200.0,200.065193,...,0.770662,0.830402,0.001576,0.831977,0.436383,0.362404,0.031615,0.001576,0.001576,0.0
